# 04 · Hardware allocation: choosing a board

Every notebook so far ran on the Brian2 simulator. This short one is about the *other* side of
the `USE_HARDWARE` switch: when you run on real hardware, **which board** does your network go
to, and how do you pick a specific one?

**By the end you'll know:**

- the two board classes and when to use each,
- why you should usually let the system choose for you,
- how to target a specific board by **machine name**, **ethernet IP**, or **board id**.

> **Info**
>
> The board calls below allocate *real* hardware, so they only run on a machine connected to a SpiNNaker2 system. On the simulator there's nothing to select — you just use [`brian2_sim.Brian2Backend()`](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/brian2_sim-api.html#spinnaker2.brian2_sim.Brian2Backend).

## 1. The two board classes

py-spinnaker2 has one class per board type, and both expose the same [`.run(net, timesteps)`](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/hardware-api.html#spinnaker2.hardware.Hardware.run) you
already know:

- **[`hardware.SpiNNaker2Chip`](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/hardware-api.html#spinnaker2.hardware.SpiNNaker2Chip)** — a single-chip board (e.g. a SpiNNode).
- **[`hardware.SpiNNcloud48NodeBoard`](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/hardware-api.html#spinnaker2.hardware.SpiNNcloud48NodeBoard)** — a 48-chip board inside a larger system.

Which selection arguments does each accept? Ask them directly:

In [1]:
import inspect
from spinnaker2 import hardware

for cls in (hardware.SpiNNaker2Chip, hardware.SpiNNcloud48NodeBoard):
    # the selection-related arguments (skip 'self' and the advanced backend option)
    args = [p for p in inspect.signature(cls.__init__).parameters if p not in ("self", "experiment_backend_type")]
    print(f"{cls.__name__:24s}: {args}")

INFO:     Path to spinnaker configuration directory found in global config path /etc/opt/spinnaker/SPINNAKER_CONFIG_PATH


SpiNNaker2Chip          : ['eth_ip', 'machine_name']
SpiNNcloud48NodeBoard   : ['eth_ip', 'machine_name', 'board_id']


So the selectors are **`machine_name`**, **`eth_ip`**, and — for the 48-chip board only —
**`board_id`**. (`experiment_backend_type` is an advanced option; leave it at its default.)

## 2. The golden rule: let the system choose

In most cases, create a board object **with no arguments**. The software then allocates the first
available board for you — this is the fastest option and avoids clashing with other users:

```python
hw = hardware.SpiNNaker2Chip()          # first free single-chip board
# or
hw = hardware.SpiNNcloud48NodeBoard()   # first free 48-chip board

hw.run(net, timesteps)
```

Only reach for a specific board when you actually need one (e.g. a board wired to a particular
sensor, or reproducing a run on the same hardware).

## 3. Targeting a specific board

| Selector | Single-chip | 48-node | What it does |
|----------|:-----------:|:-------:|--------------|
| `machine_name` (str) | ✅ | ✅ | Pick a board / frame by its system name — **preferred** |
| `eth_ip` (str) | ✅ | ✅ | Connect by ethernet IP. Works, but **discouraged** in favour of `machine_name`. |
| `board_id` (int) | ❌ | ✅ | Choose a board *index* within a named multi-node frame. |

```python
# by machine name (recommended when you need a specific board)
hw = hardware.SpiNNaker2Chip(machine_name="b201_0")

# a specific board within a multi-node frame
hw = hardware.SpiNNcloud48NodeBoard(machine_name="frame_1", board_id=0)

# by ethernet IP (discouraged, but supported)
hw = hardware.SpiNNaker2Chip(eth_ip="<board-ip>")
```

`board_id` is only applicable to 48-node boards, which are the only board type supported in multi-board frames.
Valid values range from `0` to `N-1`, where `N` is the number of boards in the frame.

## 4. Finding machine names

Machine names come from your system's SpiNNaker configuration file. Its location is stored in
`/etc/opt/spinnaker/SPINNAKER_CONFIG_PATH`, which points at a `spinnaker2_network_config.yml`
listing every machine, its type (single-chip vs 48-node), and its board count.

The [hardware-allocation guide](https://spinnaker2.gitlab.io/py-spinnaker2/advanced_topic/hardware-allocation.html)
includes a ready-made `spinnaker_boards_summary` shell function that prints that list — drop it
into your `.bashrc` to look up names quickly.

## Recap & what's next

- Two board classes: `SpiNNaker2Chip` (single-chip) and `SpiNNcloud48NodeBoard` (48-chip) — same
  `.run()` as the simulator.
- **Default to no arguments** so the system allocates the first free board.
- To specify an individual board: `machine_name` (preferred), `eth_ip` (discouraged), or `board_id` (48-node only).

**Next up — [05 · Debugging & reading the output](./05_debugging.ipynb):** turning on debug logs,
understanding what happens when you call `run()`, and the most common issues (and fixes).